# Cross-lingual PROBE transfer — the GPU session (LP4FM)

**Runtime → GPU (L4). Run all.** One session, roughly an hour.

This is the only GPU step LP4FM needs. Everything else — the surface
baselines, the renaming conditions, the exporter — is already run and
committed in `results/lp4fm/`.

**What it produces.** The 18 probe cells that currently read `not run`. Until
they exist the paper has a model-free lower bound and no claim about what the
model adds.

**The number to beat is 0.95, not majority chance.** A character n-gram model
with the variable name masked already transfers at up to 0.979 across these
languages. If the probe lands near that, cross-lingual role transfer is
surface regularity; if it clears it, there is something in the residual
stream that the surface does not carry. Either result is the paper.

**Costs, computed rather than guessed.** Extraction caches one forward per
program, so the GPU work is ~8,700 forwards, not one per occurrence. Store
size is what actually binds:

| | uncapped | capped (cell 3) |
|---|---|---|
| Qwen2.5-Coder-1.5B, 3 languages | 3.6 GB | ~1.6 GB |
| StarCoder2-7B, 3 languages | 12.4 GB | ~5.4 GB |

Start with the 1.5B model. If the probe matches the baseline, a second model
will not change that conclusion and the GPU is better spent elsewhere.


In [ ]:
# 1 - setup
import pathlib, os
# Set BRANCH to a ref that actually contains the scripts below. main is right
# once the crosslang PRs are merged; before that, use the feature branch.
BRANCH = "main"
REPO = "/content/mech-interp"
if not pathlib.Path(REPO).exists():
    !git clone -q https://github.com/nolanlwin/mech-interp.git {REPO}
%cd {REPO}
!git fetch -q origin && git checkout -q -B {BRANCH} origin/{BRANCH} && git pull -q
!git log --oneline -1
!pip install -q transformers==5.8.0 torch numpy scikit-learn matplotlib tree_sitter \
  "tree-sitter-javascript>=0.25.0" "tree-sitter-php>=0.24.1" \
  "tree-sitter-java>=0.23.5" "tree-sitter-cpp>=0.23.4" \
  "tree-sitter-go>=0.25.0" "tree-sitter-ruby>=0.23.1"
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass
from google.colab import drive
drive.mount("/content/drive")
DEST = "/content/drive/MyDrive/mech-interp/crosslang"
!mkdir -p data/xlcost outputs/role_occ outputs/activations_xlcost outputs/crosslang {DEST}
!cp -rn {DEST}/stores/* outputs/activations_xlcost/ 2>/dev/null || true
!cp -n  {DEST}/role_occ/* outputs/role_occ/ 2>/dev/null || true
!cp -n  {DEST}/data_xlcost/* data/xlcost/ 2>/dev/null || true
# Fail here, not four cells later. A ! command that cannot find its script
# prints a traceback and CARRIES ON, so a missing file surfaces as a
# FileNotFoundError in cell 4 about an artifact cell 3 never wrote.
REQUIRED = ["scripts/cap_occurrences.py", "scripts/crosslang.py",
            "scripts/export_crosslang.py", "scripts/role_occurrences.py",
            "scripts/extract_activations.py", "scripts/baselines.py"]
missing = [f for f in REQUIRED if not pathlib.Path(f).exists()]
if missing:
    raise SystemExit(
        f"BRANCH={BRANCH!r} does not contain: {missing}. Set BRANCH to a ref "
        "that has them (the crosslang feature branch until its PR merges)."
    )

import torch
print(f"setup complete | branch={BRANCH} | cuda={torch.cuda.is_available()} "
      f"{torch.cuda.get_device_name(0) if torch.cuda.is_available() else ''}")


In [ ]:
# 2 - CONFIG
MODEL = "Qwen/Qwen2.5-Coder-1.5B"
ROLES = ["accumulator", "iterator", "index_key"]
LANGS = {"Python": "python", "Javascript": "javascript", "PHP": "php"}
SPLIT = "train"
MAX_PER_ROLE = 3000        # see the store-size table above
MODEL_SLUG = MODEL.split("/")[-1].lower().replace(".", "").replace("-", "")
print(f"{MODEL}  slug={MODEL_SLUG}\nroles={ROLES}\nlanguages={list(LANGS)}")


In [ ]:
# 3 - corpora, role occurrences, and the cap. CPU, ~10 min.
#     Restricted to problems shared by at least one language PAIR: unmatched
#     transfer confounds "roles do not transfer" with "different problems",
#     so activations for unshared programs would be extracted and never used.
import json, itertools
for L, slug in LANGS.items():
    if not pathlib.Path(f"data/xlcost/{slug}_{SPLIT}.jsonl.stats.json").exists():
        !python scripts/xlcost_data.py build --language "{L}" --split {SPLIT} --out-dir data/xlcost

ids = {slug: {json.loads(l)["problem_id"]
              for l in open(f"data/xlcost/{slug}_{SPLIT}.jsonl")} for slug in LANGS.values()}
shared = set()
for a, b in itertools.combinations(ids, 2):
    shared |= ids[a] & ids[b]
print(f"problems shared by at least one pair: {len(shared)}")

for L, slug in LANGS.items():
    sub = f"data/xlcost/{slug}_{SPLIT}_shared.jsonl"
    with open(sub, "w") as out:
        for line in open(f"data/xlcost/{slug}_{SPLIT}.jsonl"):
            if json.loads(line)["problem_id"] in shared:
                out.write(line)
    occ = f"outputs/role_occ/all_{slug}_{SPLIT}.jsonl"
    if not pathlib.Path(occ + ".stats.json").exists():
        !python scripts/role_occurrences.py extract --input {sub} --role all --output {occ}
    capped = f"outputs/role_occ/capped_{slug}_{SPLIT}.jsonl"
    # ROLES_ARG is built in Python, not inlined as {" ".join(ROLES)}: IPython
    # evaluates braces inside ! commands, and nested quotes there are fragile.
    ROLES_ARG = " ".join(ROLES)
    # The marker proves the cap FINISHED; it does not prove it ran with the
    # settings above. A Drive-restored cap from an earlier session with a
    # different MAX_PER_ROLE or ROLES would otherwise be reused silently,
    # and the probe would run on the old selection while the notebook
    # reported the new one. Compare the recorded settings and re-cap on any
    # mismatch.
    stale = True
    mk = pathlib.Path(capped + ".stats.json")
    if mk.exists():
        prev = json.loads(mk.read_text())
        # Markers written before roles became a list stored the string "all"
        # for "every role"; normalise so a restored one is not read as a
        # three-element list of characters and treated as forever-stale.
        prev_roles = prev.get("roles")
        prev_roles = [] if prev_roles == "all" else list(prev_roles or [])
        stale = (prev.get("max_per_role") != MAX_PER_ROLE
                 or prev_roles != list(ROLES))
        if stale:
            print(f"  {slug}: cap settings changed "
                  f"({prev.get('roles')}, {prev.get('max_per_role')}) -> "
                  f"({ROLES}, {MAX_PER_ROLE}); invalidating everything downstream")
            # The chain is cap -> store -> probe/baseline JSONs -> table, and
            # cells 4, 5 and 6 all skip on mere existence. Clearing only part
            # of it relocates the inconsistency instead of removing it: the
            # exporter would publish results computed from the superseded
            # sample while this notebook reported the new configuration.
            # A language invalidates every cell it appears in, as SOURCE or
            # TARGET, not just the ones it trains.
            import glob as _glob
            import shutil as _shutil
            mk.unlink()
            store_dir = pathlib.Path(
                f"outputs/activations_xlcost/{slug}_{SPLIT}_{MODEL_SLUG}")
            if store_dir.exists():
                _shutil.rmtree(store_dir)
            stale_results = [
                f for pat in (f"outputs/crosslang/*_{slug}_to_*.json",
                              f"outputs/crosslang/*_to_{slug}*.json")
                for f in _glob.glob(pat)
            ]
            for f in stale_results:
                pathlib.Path(f).unlink()
            print(f"    removed store and {len(stale_results)} result file(s) "
                  f"mentioning {slug}")
    if stale:
        !python scripts/cap_occurrences.py --input {occ} --output {capped} \
          --roles {ROLES_ARG} --max-per-role {MAX_PER_ROLE}

# Every ! above can fail without stopping the cell, so check the artifacts
# rather than trusting that the commands ran.
absent = [f"outputs/role_occ/capped_{slug}_{SPLIT}.jsonl"
          for slug in LANGS.values()
          if not pathlib.Path(f"outputs/role_occ/capped_{slug}_{SPLIT}.jsonl").exists()]
if absent:
    raise SystemExit(f"capping produced nothing for: {absent}. Read the output "
                     "above for the real error; do not run cell 4.")
for slug in LANGS.values():
    st = json.loads(open(f"outputs/role_occ/capped_{slug}_{SPLIT}.jsonl.stats.json").read())
    print(f"  {slug}: {st['occurrences_out']} occurrences, "
          f"{st['problems_out']} problems, per-role {st['per_role']}")


In [ ]:
# 4 - THE GPU STEP: activation stores, one per language. ~30-45 min.
#     --label-field role is NOT optional. The store has one label slot named
#     occurrence_type; role_occurrences writes `role`. Without the flag every
#     stored label is null and probe.py drops every record -- after the GPU
#     time is spent. The choice is stamped into meta.json, and resuming with a
#     different value is refused.
for L, slug in LANGS.items():
    store = f"outputs/activations_xlcost/{slug}_{SPLIT}_{MODEL_SLUG}"
    print(f"\n######## {L} ########")
    !python scripts/extract_activations.py run \
      --canonical data/xlcost/{slug}_{SPLIT}_shared.jsonl \
      --occurrences outputs/role_occ/capped_{slug}_{SPLIT}.jsonl \
      --model-id {MODEL} --label-field role \
      --out-dir {store} --log-every 500
    !mkdir -p {DEST}/stores && cp -r {store} {DEST}/stores/ 2>/dev/null || true

bad = [f"outputs/activations_xlcost/{slug}_{SPLIT}_{MODEL_SLUG}"
       for slug in LANGS.values()
       if not pathlib.Path(
           f"outputs/activations_xlcost/{slug}_{SPLIT}_{MODEL_SLUG}/meta.json").exists()]
if bad:
    raise SystemExit(f"no store written for: {bad}. Read the output above; "
                     "cell 5 has nothing to probe.")
print("\nall stores present")


In [ ]:
# 5 - the probe transfer matrix. CPU once the stores exist.
import itertools
for role in ROLES:
    for a, b in itertools.permutations(LANGS.values(), 2):
        sa = f"outputs/activations_xlcost/{a}_{SPLIT}_{MODEL_SLUG}"
        sb = f"outputs/activations_xlcost/{b}_{SPLIT}_{MODEL_SLUG}"
        # MODEL_SLUG is in the filename because a probe result depends on the
        # model, unlike the baselines. Without it, changing MODEL rebuilds the
        # stores and then reuses the previous model's scores, and the exporter
        # publishes them against the new stores.
        out = f"outputs/crosslang/probe_{role}_{a}_to_{b}_{MODEL_SLUG}.json"
        if pathlib.Path(out).exists():
            continue
        print(f"\n=== {role}: {a} -> {b}")
        !python scripts/crosslang.py run --train-store {sa} --test-store {sb} \
          --role {role} --output {out}

import glob
made = glob.glob(f"outputs/crosslang/probe_*_{MODEL_SLUG}.json")
if not made:
    raise SystemExit("no probe results were written. Read the output above; "
                     "cell 6 would export a table with an empty probe column.")
print(f"\n{len(made)} probe result(s) written")


In [ ]:
# 6 - surface baselines on the SAME capped occurrences, then export.
#     This is not optional plumbing. The committed results/lp4fm/ baselines
#     were computed on the UNCAPPED occurrence files; the probe above runs on
#     the capped ones. Comparing those two halves would put a probe and a
#     baseline from different samples on the same row. So the baselines are
#     recomputed here from the identical files the stores were built from,
#     and the whole table is regenerated together.
#
#     CPU, ~10 min. It also gives the exporter its out_*.json inputs, without
#     which it has no rows to attach the probe columns to and exits.
import itertools
for role in ROLES:
    for a, b in itertools.permutations(LANGS.values(), 2):
        out = f"outputs/crosslang/out_{role}_{a}_to_{b}.json"
        if pathlib.Path(out).exists():
            continue
        !python scripts/baselines.py transfer \
          --train-occurrences outputs/role_occ/capped_{a}_{SPLIT}.jsonl \
          --train-canonical   data/xlcost/{a}_{SPLIT}_shared.jsonl \
          --test-occurrences  outputs/role_occ/capped_{b}_{SPLIT}.jsonl \
          --test-canonical    data/xlcost/{b}_{SPLIT}_shared.jsonl \
          --label-field role --role {role} --matched \
          --output {out}

# This cell computes the `original` condition only. The C1/C2/C4 renaming
# cells are NOT regenerated here, and the exporter rewrites the whole table
# from the inputs present -- which is how they were lost once already. They
# now live in results/lp4fm/summary_renaming_uncapped.csv, and the exporter
# refuses to drop published rows unless passed --allow-drop.
!python scripts/export_crosslang.py --in outputs/crosslang --out results/lp4fm --model {MODEL_SLUG}

from IPython.display import Image, display
import glob
print(open("results/lp4fm/SUMMARY.md").read())
for f in sorted(glob.glob("results/lp4fm/heatmap_*.png")):
    print(f); display(Image(f))

print("\nNOTE: this regenerates the WHOLE table from the capped sample, so the")
print("baseline numbers will differ from the previously committed uncapped run.")
print("That is intended -- probe and baseline must share a sample. Update the")
print("figures quoted in lp4fm_paper/section_results.tex from this table.")


In [ ]:
# 7 - save to Drive, and optionally push results to GitHub.
!mkdir -p {DEST}/stores {DEST}/role_occ {DEST}/data_xlcost {DEST}/crosslang
!cp -r outputs/activations_xlcost/* {DEST}/stores/ 2>/dev/null || true
!cp outputs/role_occ/* {DEST}/role_occ/ 2>/dev/null || true
!cp data/xlcost/*_{SPLIT}*.jsonl* {DEST}/data_xlcost/ 2>/dev/null || true
!cp outputs/crosslang/*.json {DEST}/crosslang/ 2>/dev/null || true
print(f"artifacts saved to {DEST}")

# Optional push. GH_TOKEN = fine-grained PAT, this repo only, Contents R/W.
from google.colab import userdata
try:
    tok = userdata.get("GH_TOKEN")
except Exception:
    tok = None
if not tok:
    print("No GH_TOKEN secret - download results/lp4fm/ and commit locally.")
else:
    os.environ["GH_TOKEN"] = tok
    !git config user.email "naingoolwin.astrio@gmail.com"
    !git config user.name "naingoolwin"
    !git add results/lp4fm
    !git commit -q -m "Cross-lingual probe transfer: {MODEL_SLUG}" || echo "nothing to commit"
    !git push -q https://$GH_TOKEN@github.com/nolanlwin/mech-interp.git HEAD:main && echo "pushed"
